> PostgreSQL 14+ practice. Run `sql/00_sample_data.sql` first, then `SET search_path TO sql_practice, public;` in this connection. Run cells in order in the same SQL session. See README.md for setup and expected results. These are SQL cells; the original Python notebook metadata did not make them executable as Python.


## Stored procedures with input parameters

### Business requirement: Create a procedure that updates the status of a particular transaction in the warehouse.

In [ ]:
-- Define a procedure that persists a status change in the base table; no row is inserted for an unknown ID.
CREATE OR REPLACE PROCEDURE sp_update_transaction_status(
    IN p_transaction_id BIGINT, IN p_new_status VARCHAR
)
LANGUAGE plpgsql AS $$
BEGIN
    UPDATE payment_transactions
    SET status = p_new_status
    WHERE transaction_id = p_transaction_id;
END;
$$;

## Executing the procedure

In [ ]:
-- CALL executes the procedure; the SELECT displays the stored change to payment 105.
CALL sp_update_transaction_status(105, 'Success');
SELECT * FROM payment_transactions WHERE transaction_id = 105;

## Stored procedures with local variables

### Business requirement: Calculate the number of successful transactions and their total payment volume.

In [ ]:
-- SELECT INTO assigns aggregate values to local variables; RAISE NOTICE prints the summary.
CREATE OR REPLACE PROCEDURE sp_payment_summary()
LANGUAGE plpgsql AS $$
DECLARE
    v_total_transaction BIGINT;
    v_total_amount NUMERIC;
BEGIN
    SELECT COUNT(*), COALESCE(SUM(amount), 0)
    INTO v_total_transaction, v_total_amount
    FROM payment_transactions
    WHERE status = 'Success';
    RAISE NOTICE 'Successful payment_transactions: %, total amount: %', v_total_transaction, v_total_amount;
END;
$$;
CALL sp_payment_summary();

## Stored procedures with output parameters

### Business requirement: Return payment summary metrics to an external application rather than simply printing them.

In [ ]:
-- Typed OUT parameters return successful count and amount for one customer.
CREATE OR REPLACE PROCEDURE sp_customer_summary(
    IN p_customer_id BIGINT, OUT p_total_transaction BIGINT, OUT p_total_amount NUMERIC
)
LANGUAGE plpgsql AS $$
BEGIN
    SELECT COUNT(*), COALESCE(SUM(amount), 0)
    INTO p_total_transaction, p_total_amount
    FROM payment_transactions
    WHERE customer_id = p_customer_id AND status = 'Success';
END;
$$;

## Exccute thr procedure

In [ ]:
-- Pass NULL placeholders for OUT parameters; the result row contains the calculated metrics.
CALL sp_customer_summary(201, NULL, NULL);